# Optimal Execution Visual Lab

## Executive summary
Execution speed exchanges market impact for timing risk. This notebook uses one reproducible synthetic market to connect Almgren–Chriss scheduling, resilient liquidity, a reactive limit-order book, transaction-cost analysis, and bounded reinforcement-learning adjustments. Economic implementation shortfall is kept separate from the RL training reward.

### 日本語サマリー
執行を速めると価格変動リスクは下がりますが、市場インパクトは上がります。本ノートブックは、Almgren–Chriss、板の回復力、反応型板、取引コスト分析、制約付き強化学習を、同一の再現可能な合成市場で比較します。RLの整形報酬と経済的な実装ショートフォールは明確に分離します。

In [ ]:
import os
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

from optimal_execution.config import load_config
from optimal_execution.provenance import artifact_dirs

config_path = Path(os.environ.get("OPTIMAL_EXECUTION_CONFIG", "configs/quick.yaml"))
cfg = load_config(config_path)
paths = artifact_dirs(cfg)
display(
    Markdown(
        f"**Profile:** `{cfg.profile}` · **seed:** `{cfg.seed}` · **order:** `{cfg.initial_inventory:,.0f}` shares · **horizon:** `{cfg.horizon_seconds:.0f}` seconds"
    )
)

## Concept map and market-microstructure definitions

**Spread** is the immediate cost of crossing the best quotes. **Liquidity** combines displayed depth and replenishment. **Market impact** is the agent's own price footprint. **Timing risk** is the uncertainty from holding inventory while the unaffected price moves. An **execution policy** maps time, inventory, and book state into bounded market and limit orders.

Strategic baseline (AC / resilient schedule) → tactical heuristic or RL adjustment → safety layer (inventory, participation, price collar, deadline).

In [ ]:
display(
    pd.DataFrame(
        {
            "variable": [
                "side",
                "arrival price",
                "initial inventory",
                "steps",
                "annualized volatility",
                "ADV",
                "spread",
                "participation cap",
            ],
            "value": [
                cfg.side,
                cfg.arrival_price,
                cfg.initial_inventory,
                cfg.n_decision_steps,
                cfg.annualized_volatility,
                cfg.average_daily_volume,
                cfg.spread_bps,
                cfg.max_participation_rate,
            ],
            "unit": [
                "",
                "price/share",
                "shares",
                "",
                "fraction/year",
                "shares/day",
                "bps",
                "fraction",
            ],
        }
    )
)

## Synthetic market profiles and unaffected price paths
Volume, volatility, spread, and depth follow documented intraday profiles with local stochastic multipliers. The unaffected price contains no agent impact; impacted prices are layered on top. Common random numbers make strategy differences paired rather than independent.

In [ ]:
display(Image(filename=str(paths["figures"] / "unaffected_price_paths.png")))

## Temporary, permanent, transient, and square-root impact
Temporary impact is paid only while trading. Permanent impact shifts later prices. Transient impact decays after trading according to resilience. The square-root impact law is a separate empirical diagnostic and is not added to the linear channels.

In [ ]:
for name in ["impact_channels.png", "impact_recovery.png", "sqrt_impact.png"]:
    display(Image(filename=str(paths["figures"] / name)))

## Almgren–Chriss derivation, sensitivities, and efficient frontier
For objective `integral[eta * xdot^2 + lambda * sigma^2 * x^2] dt`, urgency is `kappa = sqrt(lambda * sigma^2 / eta)`. The inventory trajectory is proportional to `sinh(kappa * (T-t))`; as `kappa → 0`, it becomes the linear TWAP path. Higher risk aversion or volatility front-loads execution, while higher temporary impact spreads it out. Order size scales quantity but not the normalized trajectory under this linear specification.

In [ ]:
display(Image(filename=str(paths["figures"] / "ac_trajectories.png")))
display(Image(filename=str(paths["figures"] / "efficient_frontier.png")))

## Obizhaeva–Wang-style resilient liquidity
The project implements the exact risk-neutral continuous exponential-resilience schedule and a discrete convex quadratic optimizer. It does not claim an exact risk-averse Obizhaeva–Wang solution. Low resilience leaves persistent displacement; high resilience supports faster reuse of liquidity.

In [ ]:
resilience = pd.read_csv(paths["data"] / "resilience_sweep.csv")
display(resilience.groupby(["rho", "method"], as_index=False)[["cost", "twap_cost"]].first())

## Reactive limit-order book, queue imbalance, and event state
The simplified LOB tracks bid/ask depth, spread, queue imbalance, recent flow, volatility, and agent impact. Exogenous market orders, limit arrivals, cancellations, and replenishment react to state. A large agent market order consumes L1 depth, walks deeper liquidity, widens the spread, and changes later fills. This is not historical replay and is not exchange-grade realism.

In [ ]:
display(Image(filename=str(paths["figures"] / "lob_depth.png")))
display(Image(filename=str(paths["figures"] / "queue_imbalance.png")))

## Market-order walk, limit-order queue, fill probability, and adverse selection
Market orders buy immediacy but pay spread and book-walk costs. Passive orders can capture spread, but queue ahead lowers fill probability and deadline cleanup can dominate. Latent short-horizon alpha jointly tilts order flow and the next price move, creating stylized adverse selection.

In [ ]:
display(Image(filename=str(paths["figures"] / "market_vs_limit.png")))
lob_summary = pd.read_csv(paths["metrics"] / "lob_strategy_summary.csv")
display(
    lob_summary[["strategy_id", "is_mean_bps", "cvar95_bps", "fill_rate", "cleanup_qty"]].round(3)
)

## Static schedules, implementation-shortfall distributions, TCA decomposition, and tail risk
Immediate, TWAP, VWAP-style, POV, Almgren–Chriss, and resilience-aware schedules share identical stochastic paths. Positive implementation shortfall is adverse for both sells and buys. Latent timing, spread, temporary, permanent, transient, fee, and cleanup components are exact only inside this synthetic model.

In [ ]:
for name in ["strategy_schedules.png", "is_distributions.png", "tca_decomposition.png"]:
    display(Image(filename=str(paths["figures"] / name)))
classical = pd.read_csv(paths["metrics"] / "classical_strategy_summary.csv")
display(
    classical[
        ["strategy_id", "is_mean_bps", "is_mean_ci_lo_bps", "is_mean_ci_hi_bps", "cvar95_bps"]
    ].round(3)
)

## Reactive versus non-reactive simulation
The control run reuses the same exogenous draws but removes the agent's footprint. The resulting cost gap quantifies what this toy replay understates for the configured order size; it is not an estimate for a real venue.

In [ ]:
display(Image(filename=str(paths["figures"] / "reactive_vs_replay.png")))
display(
    pd.read_csv(paths["metrics"] / "reactive_comparison.csv")[
        ["mode", "is_mean_bps", "cvar95_bps"]
    ].round(3)
)

## RL state, actions, reward, safety layer, and training curves
The 12 normalized observations include time, inventory, spread, both depths, imbalance, recent return and flow, volatility, transient impact, volume, and outstanding limit quantity. A 15-action grid combines bounded market-size multipliers with no-order, passive, or improving-limit directives. PPO trains on shaped incremental cost and inventory/impact penalties, while all comparisons use economic implementation shortfall. Hard constraints cap child size and participation, prevent over-execution, apply a price collar, and force deadline liquidation.

In [ ]:
display(Image(filename=str(paths["figures"] / "rl_training_history.png")))
history = pd.read_csv(paths["metrics"] / "rl_training_history.csv")
display(
    history.groupby("run_id", as_index=False)
    .tail(1)[["run_id", "episodes", "train_is_ma_bps", "best_val_is_bps"]]
    .round(3)
)

## In-distribution comparison and stress tests
TWAP, Almgren–Chriss, the mixed heuristic, residual RL, and free RL are tested on fixed held-out seeds. Stress regimes change volatility, liquidity, volume profile, adverse alpha, and spread/resilience. The quick profile has one RL seed, so it cannot establish seed-robust superiority.

In [ ]:
display(Image(filename=str(paths["figures"] / "stress_test_heatmap.png")))
stress = pd.read_csv(paths["metrics"] / "stress_summary.csv")
display(stress.pivot(index="strategy_id", columns="regime", values="is_mean_bps").round(3))

## Feature ablation, RL action diagnostics, and distribution shift
Each residual policy is retrained after masking one observation group. The model-misspecification test keeps the trained policy fixed while changing resilience, depth, spread stress, and liquidity. Differences are simulator evidence, not causal feature importance in real markets.

In [ ]:
display(Image(filename=str(paths["figures"] / "feature_ablation.png")))
display(Image(filename=str(paths["figures"] / "model_misspecification.png")))
display(
    pd.read_csv(paths["metrics"] / "ablation_summary.csv")[
        ["strategy_id", "feature_removed", "is_mean_bps", "delta_vs_full_bps"]
    ].round(3)
)

## What the experiment establishes
It establishes that the implementation is reproducible under recorded seeds; classical urgency responds correctly to risk and impact parameters; the agent changes its synthetic market; limit orders exchange spread for fill risk; and policy rankings can change under stress and model shift.

In [ ]:
best_classical = classical.loc[classical.is_mean_bps.idxmin()]
best_lob = lob_summary.loc[lob_summary.is_mean_bps.idxmin()]
id_test = stress[stress.regime == "in_distribution"]
best_id = id_test.loc[id_test.is_mean_bps.idxmin()]
display(
    Markdown(f"""**Generated quick-run findings**  
- Lowest classical mean cost: `{best_classical.strategy_id}` at **{best_classical.is_mean_bps:.3f} bps**.  
- Lowest reactive-LOB mean cost: `{best_lob.strategy_id}` at **{best_lob.is_mean_bps:.3f} bps**.  
- Lowest in-distribution comparison cost: `{best_id.strategy_id}` at **{best_id.is_mean_bps:.3f} bps**.  
These rankings are conditional on the synthetic calibration.""")
)

## What it does not establish; limitations and next steps
This lab does not establish real-market profitability, causal feature value, exchange realism, or statistical RL superiority. It omits hidden liquidity, latency, multi-venue routing, strategic counterparties, manipulation constraints, cross-impact, and empirical calibration. Next steps are real-data calibration with strict holdouts, multi-asset cross-impact, dealer/RFQ markets, stochastic resilience, safer constrained RL, distributionally robust evaluation, and interaction with rough volatility or Hawkes flow.